## 🎯 Learning Objectives
* Identify common failure modes in initial LLM prompts.
* Apply systematic prompt engineering techniques (e.g., clear instructions, role-playing, few-shot, chain-of-thought, output formatting) to improve prompt performance.
* Iteratively refine prompts based on specific evaluation criteria and desired outcomes.
* Understand the importance of explicit constraints and negative instructions in prompt design.


## Exercise: Systematically Improve a Failing Prompt

**Lesson ID:** LLM03-L05

In this exercise, you will act as a Prompt Engineer for "AgenticHealth," a cutting-edge AI-powered healthcare platform. Your task is to improve a failing prompt designed to summarize patient medical notes. The initial prompt often produces verbose summaries, misses critical information, or, most critically, fails to redact sensitive Personally Identifiable Information (PII).

### Scenario
AgenticHealth uses LLMs to generate concise summaries of patient medical notes for doctors to quickly review. These summaries must be:
1.  **Concise**: No more than 150 words.
2.  **Structured**: Include sections for 'Key Diagnoses', 'Medications', 'Treatment Plan', and 'Next Steps'.
3.  **Accurate**: Capture all critical medical information.
4.  **Secure**: Absolutely *no* Personally Identifiable Information (PII) such as patient names, addresses, or specific dates of birth should be present in the summary. Replace any PII with `[REDACTED]`. 

### The Failing Prompt
Your team has provided an initial, very basic prompt that consistently fails to meet the above requirements:

```
Summarize the following patient medical note:

{medical_note}
```

### Your Task
Your goal is to systematically improve this prompt. You should iterate on the prompt, applying various prompt engineering techniques learned in previous lessons, until it consistently produces a summary that meets all the requirements.

**Steps:**
1.  **Analyze the Failure**: Understand *why* the initial prompt fails based on the requirements.
2.  **Iterate and Refine**: Modify the prompt step-by-step. For each iteration, consider:
    *   Adding clear instructions.
    *   Defining a specific role for the LLM.
    *   Specifying output format (e.g., JSON, bullet points, sections).
    *   Using few-shot examples (if beneficial, though not strictly required for this exercise).
    *   Implementing chain-of-thought reasoning (e.g., "First, identify PII. Then, redact it. Finally, summarize.").
    *   Adding negative constraints (e.g., "Do NOT include PII").
    *   Specifying length constraints.
3.  **Test**: Use the provided `LLM_API_CALL` mock function to test your prompt against the sample medical note.
4.  **Document**: Explain the changes you made in each iteration and why you believe they improve the prompt.

### Evaluation Criteria
Your improved prompt will be evaluated on its ability to generate a summary that is:
*   **Concise**: Under 150 words.
*   **Structured**: Contains 'Key Diagnoses', 'Medications', 'Treatment Plan', and 'Next Steps' sections.
*   **Accurate**: Reflects the core information from the medical note.
*   **PII-Free**: All PII is correctly redacted with `[REDACTED]`.
*   **Robust**: The prompt should be general enough to work with similar medical notes.

Good luck, Prompt Engineer!


In [ ]:
import json
import re

# --- Mock LLM API Call Function (Simulates LLM behavior) ---
# In a real scenario, this would be an actual API call to OpenAI, Anthropic, Google, etc.
# For this exercise, it simulates responses to help you iterate.

def LLM_API_CALL(prompt: str, model: str = "gpt-4o-2026-05-13") -> str:
    """
    A mock function to simulate an LLM API call.
    It returns a predefined 'bad' response for the initial prompt
    and a 'good' response if the prompt contains specific keywords
    indicating improvement (for demonstration purposes).
    """
    print(f"\n--- Calling Mock LLM with model: {model} ---")
    print(f"Prompt snippet: {prompt[:200]}...")

    # Simulate a 'bad' initial response for a very basic prompt
    if "summarize the following patient medical note:" in prompt.lower() and \
       "key diagnoses" not in prompt.lower() and \
       "redact" not in prompt.lower():
        print("\n--- Mock LLM Response (Initial Failing Prompt) ---")
        return (
            "Dr. Smith's patient, Jane Doe, 45, residing at 123 Main St, Anytown, was seen on 2026-03-10. "
            "She presented with severe abdominal pain. Diagnosed with acute appendicitis. "
            "Prescribed Ibuprofen 400mg. Surgery scheduled for 2026-03-12. Follow-up with Dr. Smith in 2 weeks. "
            "Her contact is 555-123-4567. She mentioned her birthday is 1981-07-20."
        )

    # Simulate an improved response if the prompt includes key instructions
    if "key diagnoses" in prompt.lower() and \
       "medications" in prompt.lower() and \
       "redact" in prompt.lower() and \
       "no pii" in prompt.lower() and \
       "structured format" in prompt.lower():
        print("\n--- Mock LLM Response (Improved Prompt) ---")
        return (
            "**Key Diagnoses:** Acute appendicitis, mild dehydration.\n\n"
            "**Medications:** Ibuprofen 400mg (as needed for pain), IV fluids (during hospital stay).\n\n"
            "**Treatment Plan:** Emergency appendectomy performed on [REDACTED]. Post-operative care includes pain management and monitoring for complications.\n\n"
            "**Next Steps:** Follow-up appointment with surgical team in 2 weeks. Monitor incision site for infection. Resume normal activities gradually. Patient advised to contact clinic if fever or increased pain occurs."
        )

    # Default response for other prompts (e.g., if student tries something else)
    print("\n--- Mock LLM Response (Default) ---")
    return "I'm sorry, I couldn't generate a summary that meets all the complex requirements with the given prompt. Please refine your instructions."

# --- Sample Patient Medical Note ---
medical_note = """
Patient Name: Jane Doe
Date of Birth: 1981-07-20
Address: 123 Main St, Anytown, USA
Contact: 555-123-4567
Attending Physician: Dr. Emily Smith
Date of Visit: 2026-03-10

**Chief Complaint:** Severe abdominal pain, onset 24 hours prior, localized to the right lower quadrant.

**History of Present Illness:** Ms. Doe, a 45-year-old female, presented to the emergency department with acute, sharp abdominal pain. Pain score 8/10. Associated with nausea and loss of appetite. No fever or vomiting initially, but developed mild fever (100.2 F) in ED. Last menstrual period 2 weeks ago. No significant past medical history. No known allergies.

**Physical Examination:**
-   **General:** Alert and oriented x3, appears distressed.
-   **Abdomen:** Tenderness and guarding in the right lower quadrant, positive rebound tenderness. Bowel sounds present but diminished.
-   **Vitals:** BP 120/80, HR 98, RR 18, Temp 100.2 F.

**Labs:** WBC 15.2 (elevated), CRP 12 (elevated). Urinalysis negative.

**Imaging:** CT abdomen/pelvis showed an inflamed appendix, consistent with acute appendicitis. No signs of perforation.

**Assessment:** Acute appendicitis.

**Plan:**
1.  Admit to surgical service.
2.  NPO (nothing by mouth).
3.  IV fluids (Lactated Ringer's).
4.  IV antibiotics (Cefoxitin 1g IV q8h).
5.  Pain management (Morphine 2mg IV PRN).
6.  Emergency appendectomy scheduled for 2026-03-12 at 08:00 AM.
7.  Post-op care and monitoring.
8.  Follow-up with Dr. Smith in 2 weeks post-discharge.
"""

# --- Initial Failing Prompt (for reference) ---
initial_failing_prompt = f"Summarize the following patient medical note:\n\n{medical_note}"

print("\n--- Initial Failing Prompt Output (for reference) ---")
initial_response = LLM_API_CALL(initial_failing_prompt)
print(initial_response)

# --- Helper function to evaluate response length ---
def count_words(text: str) -> int:
    return len(re.findall(r'\b\w+\b', text))

# --- Helper function to check for PII (simple regex for common patterns) ---
def contains_pii(text: str) -> bool:
    pii_patterns = [
        r'Jane Doe', r'1981-07-20', r'123 Main St', r'Anytown', r'555-123-4567',
        r'Dr. Emily Smith', r'2026-03-10', r'2026-03-12'
    ]
    for pattern in pii_patterns:
        if re.search(pattern, text, re.IGNORECASE):
            return True
    return False

print(f"\nInitial response word count: {count_words(initial_response)}")
print(f"Initial response contains PII: {contains_pii(initial_response)}")


### Your Implementation

Now it's your turn! In the code cell below, implement your improved prompt. You should:

1.  Define your `improved_prompt` variable.
2.  Use the `medical_note` variable provided in the setup code.
3.  Call the `LLM_API_CALL` function with your `improved_prompt`.
4.  Print the response.
5.  Optionally, add comments to explain your thought process and the changes you made in each iteration.
6.  Use the `count_words` and `contains_pii` helper functions to evaluate your output.

Remember to iterate! Don't expect to get it perfect on the first try. Think about how you can systematically add instructions, constraints, and structure to guide the LLM.


In [ ]:
# --- Student's Workspace (Reference Solution Below) ---

# Iteration 1: Adding basic instructions and structure
# improved_prompt_v1 = f"""
# As a medical assistant, summarize the following patient medical note. 
# The summary must be concise, under 150 words, and structured into sections:
# Key Diagnoses, Medications, Treatment Plan, and Next Steps.
# Do NOT include any Personally Identifiable Information (PII). Replace PII with [REDACTED].

# Medical Note:
# {medical_note}
# """
# response_v1 = LLM_API_CALL(improved_prompt_v1)
# print(f"\n--- Improved Prompt V1 Response ---")
# print(response_v1)
# print(f"Word count: {count_words(response_v1)}")
# print(f"Contains PII: {contains_pii(response_v1)}")

# Iteration 2: Refining instructions, adding explicit output format
# improved_prompt_v2 = f"""
# You are an expert medical summarizer for AgenticHealth. Your task is to create a concise, structured summary of the patient's medical note.
# 
# **Instructions:**
# 1.  **Identify and Redact PII**: Before summarizing, identify all Personally Identifiable Information (PII) such as names, dates of birth, addresses, contact numbers, and specific dates (except general visit date if relevant to the summary, but redact specific visit dates if they are PII). Replace all identified PII with the placeholder `[REDACTED]`.
# 2.  **Summarize Concisely**: The final summary must be under 150 words.
# 3.  **Structure the Output**: Present the summary in the following markdown format with bolded headings:
#     **Key Diagnoses:** [List diagnoses]
#     **Medications:** [List medications]
#     **Treatment Plan:** [Describe treatment plan]
#     **Next Steps:** [Outline next steps]
# 4.  **Accuracy**: Ensure all critical medical details are accurately reflected.
# 
# **Patient Medical Note:**
# {medical_note}
# """
# response_v2 = LLM_API_CALL(improved_prompt_v2)
# print(f"\n--- Improved Prompt V2 Response ---")
# print(response_v2)
# print(f"Word count: {count_words(response_v2)}")
# print(f"Contains PII: {contains_pii(response_v2)}")


# --- Final Reference Solution ---
# This solution incorporates several best practices for systematic prompt improvement.

final_improved_prompt = f"""
As a highly skilled medical summarization AI for AgenticHealth, your primary goal is to extract and present critical patient information from the provided medical note in a structured, concise, and secure manner. 

**Strict Requirements:**
1.  **PII Redaction (CRITICAL)**: Absolutely *no* Personally Identifiable Information (PII) must appear in the summary. This includes, but is not limited to, patient names, specific dates of birth, addresses, phone numbers, specific visit dates (unless generalized), and physician names. Replace *all* PII with the exact string `[REDACTED]`. This is non-negotiable.
2.  **Conciseness**: The entire summary must be under 150 words.
3.  **Structured Output**: Present the summary using the following markdown format with bolded headings. Ensure each section is populated accurately.
    **Key Diagnoses:**
    **Medications:**
    **Treatment Plan:**
    **Next Steps:**
4.  **Accuracy**: Ensure all medical facts, diagnoses, and plans are accurately reflected from the original note.
5.  **Tone**: Maintain a professional, objective, and clinical tone.

**Chain of Thought Process (Internal to LLM - do not output this):**
1.  First, carefully read through the entire medical note to identify all key medical information (diagnoses, medications, treatment plan, next steps).
2.  Simultaneously, identify *all* instances of PII. Create a mental list of these items.
3.  Draft the summary, ensuring it flows logically and covers all required sections.
4.  During drafting, *immediately* replace any identified PII with `[REDACTED]`.
5.  Review the drafted summary for conciseness (under 150 words), structure, accuracy, and most importantly, *complete PII redaction*.
6.  If any PII remains or if the summary is too long, revise until all requirements are met.

**Patient Medical Note to Summarize:**
{medical_note}
"""

print("\n--- Final Reference Solution Prompt Output ---")
final_response = LLM_API_CALL(final_improved_prompt)
print(final_response)

# --- Evaluation of Final Response ---
print(f"\nFinal response word count: {count_words(final_response)}")
print(f"Final response contains PII: {contains_pii(final_response)}")

# --- Expected Output (for comparison) ---
expected_output_structure = """
**Key Diagnoses:** Acute appendicitis, mild dehydration.

**Medications:** Ibuprofen 400mg (as needed for pain), IV fluids (during hospital stay).

**Treatment Plan:** Emergency appendectomy performed on [REDACTED]. Post-operative care includes pain management and monitoring for complications.

**Next Steps:** Follow-up appointment with surgical team in 2 weeks. Monitor incision site for infection. Resume normal activities gradually. Patient advised to contact clinic if fever or increased pain occurs.
"""

print("\n--- Expected Output (for comparison) ---")
print(expected_output_structure)

# Additional check for structure (simple keyword check)
def check_structure(text: str) -> bool:
    required_sections = ["Key Diagnoses", "Medications", "Treatment Plan", "Next Steps"]
    return all(section in text for section in required_sections)

print(f"Final response has correct structure: {check_structure(final_response)}")

# --- Explanation of Improvements in the Final Prompt ---
# 1.  **Role Assignment**: "As a highly skilled medical summarization AI for AgenticHealth..." - Gives the LLM a clear persona and purpose.
# 2.  **Clear Goal**: "...extract and present critical patient information... structured, concise, and secure manner." - Sets the overall objective.
# 3.  **Strict Requirements Section**: Uses bolded "Strict Requirements" to emphasize critical constraints, making them stand out.
# 4.  **PII Redaction (CRITICAL)**: Explicitly states the non-negotiable nature of PII redaction, provides examples of PII, and specifies the exact replacement string `[REDACTED]`. This is crucial for security.
# 5.  **Conciseness**: Reinforces the word count limit.
# 6.  **Structured Output**: Provides a precise markdown template for the output, guiding the LLM to produce the desired format.
# 7.  **Accuracy & Tone**: Reminds the LLM about factual correctness and professional tone.
# 8.  **Chain of Thought Process**: This is a powerful technique. By outlining an internal thought process for the LLM, we guide it through the steps it should take to achieve the desired outcome (identify PII -> redact -> summarize -> review). This significantly improves reliability, especially for complex tasks like PII redaction and structured output generation. The instruction "(Internal to LLM - do not output this)" prevents the LLM from printing its thought process.
# 9.  **Clear Delimiters**: Uses markdown headings and clear separation for the medical note to distinguish instructions from input.
